**Parcial 2 : Análisis de datos del mercado laboral GEIH 2026: DESARROLLO**

1.Para el total nacional calcule (a Junio de 2026): 
- población en edad de trabajar  
- población que no está en edad de trabajar 
- población económicamente activa 
- población económicamente inactiva 
- tasa de ocupación -tasa de desempleo  

In [0]:
import pandas as pd

In [0]:
# Seleccionar el período de referencia
fuerzadetrabajo_junio = fuerzadetrabajo[fuerzadetrabajo["PERIODO"] == 20260626].copy()

In [0]:
fuerzadetrabajo_junio

In [0]:
# Calcular la población en edad de trabajar
PET = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["PET"] == 1, "FEX_C18"
].sum()

print(f"Población en edad de trabajar: {PET:,.0f} personas")


In [0]:
# Calcular la PET desde características generales
PET_CGSE = CGSE.loc[
    (CGSE["PERIODO"] == 20260626) &
    (CGSE["P6040"] >= 15), "FEX_C18"
].sum()

print(f"PET desde Fuerza de trabajo: {PET:,.0f}")
print(f"PET desde Características generales: {PET_CGSE:,.0f}")

**_R/ el total nacional de personas en edad de trabajar para junio de 2026 es de 24,095,346 personas _**

In [0]:
# Filtrar el período de junio de 2026
CGSE = spark.read.format("delta").table("workspace.electiva.CGSE").toPandas()
CGSE_junio = CGSE.loc[
    CGSE["PERIODO"] == 20260626
].copy()

In [0]:
CGSE_junio

In [0]:
# Calcular la población que no está en edad de trabajar
# En este cálculo: personas menores de 15 años
poblacion_no_PET = CGSE_junio.loc[
    CGSE_junio["P6040"] < 15, "FEX_C18"
].sum()

print(
    f"Población que no está en edad de trabajar: "
    f"{poblacion_no_PET:,.0f} personas"
)

**_**R/ El total nacional de personas que no esta en edad de trabajar para junio de 2026 es de 7,275,710 personas ``**_**

In [0]:
# Población económicamente activa (PEA)
PEA = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["FT"] == 1, "FEX_C18"
].sum()

print("Población económicamente activa:", round(PEA))


**_**R/ El total nacional de la poblacion economicamente activa para junio de 2026 es de 15,432,739 personas ``**_**

In [0]:
# Población económicamente inactiva (PEI)
PEI = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["FFT"] == 1, "FEX_C18"
].sum()

print("Población económicamente inactiva:", round(PEI))


**_**R/ El total nacional de la poblacion economicamente inactiva para junio de 2026 es de 8,662,607 personas ``**_**

In [0]:
# Filtrar el período de junio de 2026
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()
ocupados_junio = ocupados.loc[
    ocupados["PERIODO"] == 20260626
].copy()

In [0]:
ocupados_junio

In [0]:
# Cargar y filtrar fuerza de trabajo para junio de 2026
fuerzadetrabajo = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo").toPandas()
fuerzadetrabajo_junio = fuerzadetrabajo[fuerzadetrabajo["PERIODO"] == 20260626].copy()

# Población en edad de trabajar (PET)
pet = fuerzadetrabajo_junio.loc[
    fuerzadetrabajo_junio["PET"] == 1, "FEX_C18"
].sum()

# Población ocupada (PO)
po = ocupados_junio["FEX_C18"].sum()

# Tasa de ocupación (TO)
tasa_ocupacion = (po / pet) * 100

print(f"Población en edad de trabajar: {pet:,.0f}")
print(f"Población ocupada: {po:,.0f}")
print(f"Tasa de ocupación: {tasa_ocupacion:.2f}%")

_**R/ La tasa de ocupacion nacional para el periodo dejunio 2026 es de 59,10%**_

In [0]:
# Cargar la tabla de desempleados y filtrar junio de 2026
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados").toPandas()
desempleados_junio = desempleados[desempleados["PERIODO"] == 20260626].copy()

# Población desocupada (PD)
PD = desempleados_junio.loc[desempleados_junio["DSI"] == 1, "FEX_C18"].sum()

# Tasa de desempleo (TD)
tasa_desempleo = (PD / PEA) * 100

print(f"Población desocupada: {PD:,.0f}")
print(f"Tasa de desempleo: {tasa_desempleo:.2f}%")

_**R/ La tasa de desempleo nacional para el periodo dejunio 2026 es de 7,72%**_

2. Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?

In [0]:
import pandas as pd

In [0]:
#Leyendo la tabla de caracteristicas generales
fuerzadetrabajo = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
fuerzadetrabajo = fuerzadetrabajo.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

fuerzadetrabajo = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')


In [0]:
desempleados

In [0]:
# desempleados_dpto -> cantidad de hombres y mujeres desempleados por departamento
desempleados_dpto_genero = desempleados[desempleados['DSI']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_dpto_genero

In [0]:
#cantidad de hombres y mujeres economicamente activos por departamento
pea_dpto_genero = fuerzadetrabajo[fuerzadetrabajo['FT']==1].groupby(['P3271_x','DPTO'])['FEX_C18'].sum()

In [0]:
pea_dpto_genero

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero, 'DESEMPLEADOS':desempleados_dpto_genero})

In [0]:
tabla_desempleo

In [0]:
tabla_desempleo['TD']= (tabla_desempleo['DESEMPLEADOS']/tabla_desempleo['PEA'])*100


In [0]:
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

sexo_nombre = {1:'Hombre',2:'Mujer'}

tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['level_0'].map(sexo_nombre)


In [0]:
tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo

In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=[ 'Sexo','TD'], ascending=False)
tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo']=='Hombre']

**R/ los departamentos con mayores tasas de desempleo son:
putumayo para las mujeres y san andres para loshombres**

**y los departamentos con menores tasas de desempleo son:
bolivar para las mujeres y vaupes para los hombres**



3.  Tomando en consideración únicamente los siguientes grupos de edad:

- 18 a 24 
- 24 a 30 
- 30 a 40 
- 40 a 50  
- 60 - en adelante 

Para cada uno de los departamentos, cual es el grupo de edad con mayor y menor 
tasa de desempleo ?

In [0]:
llaves = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']

# cgse solo de junio, para que el merge no duplique filas si trae más períodos
cgse_junio = cgse[cgse['PERIODO'] == 20260626][llaves + ['P3271', 'P6040']]

In [0]:
# Anexar sexo y edad (el drop evita columnas duplicadas si la celda se ejecuta dos veces)
fuerzadetrabajo_junio = (fuerzadetrabajo_junio
                         .drop(columns=['P3271', 'P6040'], errors='ignore')
                         .merge(cgse_junio, on=llaves, how='left'))

desempleados_junio = (desempleo_j
                      .drop(columns=['P3271', 'P6040'], errors='ignore')
                      .merge(cgse_junio, on=llaves, how='left'))


In [0]:
# Grupos de edad
bins = [18, 24, 30, 40, 50, 60, 200]
etiquetas = ['18 a 24', '24 a 30', '30 a 40', '40 a 50', '50 a 60', '60 en adelante']

fuerzadetrabajo_junio['GRUPO_EDAD'] = pd.cut(
    fuerzadetrabajo_junio['P6040'], bins=bins, labels=etiquetas, right=False)
desempleados_junio['GRUPO_EDAD'] = pd.cut(
    desempleados_junio['P6040'], bins=bins, labels=etiquetas, right=False)

In [0]:
# PEA y desempleados por departamento y grupo de edad
pea_edad = fuerzadetrabajo_junio[fuerzadetrabajo_junio['FT'] == 1].groupby(
    ['DPTO', 'GRUPO_EDAD'], observed=True)['FEX_C18'].sum()

desemp_edad = desempleados_junio[desempleados_junio['DSI'] == 1].groupby(
    ['DPTO', 'GRUPO_EDAD'], observed=True)['FEX_C18'].sum()

In [0]:
# Tabla con la tasa de desempleo
tabla_edad = pd.DataFrame({'PEA': pea_edad, 'DESEMPLEADOS': desemp_edad})
tabla_edad['DESEMPLEADOS'] = tabla_edad['DESEMPLEADOS'].fillna(0)
tabla_edad['TD'] = tabla_edad['DESEMPLEADOS'] / tabla_edad['PEA'] * 100

tabla_edad = tabla_edad.reset_index()
tabla_edad['GRUPO_EDAD'] = tabla_edad['GRUPO_EDAD'].astype(str)
tabla_edad['Departamento'] = tabla_edad['DPTO'].map(dpto_nombre)


In [0]:
# Solo los grupos que pide el taller (se descarta 50 a 60)
tabla_edad = tabla_edad[tabla_edad['GRUPO_EDAD'] != '50 a 60']
tabla_edad = tabla_edad[['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']]

In [0]:
tabla_edad

In [0]:
idx_max = tabla_edad.groupby('Departamento')['TD'].idxmax()
idx_min = tabla_edad.groupby('Departamento')['TD'].idxmin()

mayor = tabla_edad.loc[idx_max, ['Departamento', 'GRUPO_EDAD', 'TD']].rename(
    columns={'GRUPO_EDAD': 'Grupo mayor TD', 'TD': 'Mayor TD (%)'})
menor = tabla_edad.loc[idx_min, ['Departamento', 'GRUPO_EDAD', 'TD']].rename(
    columns={'GRUPO_EDAD': 'Grupo menor TD', 'TD': 'Menor TD (%)'})

resultado_edad = mayor.merge(menor, on='Departamento').sort_values('Departamento')

In [0]:
resultado_edad

4. ¿Cuáles son los 3 departamentos con menor tasa de desempleo de mujeres entre 
los 18 y los 25 años ? 

In [0]:
# Filtro: mujeres entre 18 y 25 años (ambos extremos incluidos)
mujeres_ft = fuerzadetrabajo_junio[
    (fuerzadetrabajo_junio['P3271'] == 2) &
    (fuerzadetrabajo_junio['P6040'].between(18, 25))
]

mujeres_desemp = desempleados_junio[
    (desempleados_junio['P3271'] == 2) &
    (desempleados_junio['P6040'].between(18, 25))
]

In [0]:
# PEA, desempleadas y casos de muestra por departamento
pea_mujeres = mujeres_ft[mujeres_ft['FT'] == 1].groupby('DPTO')['FEX_C18'].sum()
desemp_mujeres = mujeres_desemp[mujeres_desemp['DSI'] == 1].groupby('DPTO')['FEX_C18'].sum()
casos_pea = mujeres_ft[mujeres_ft['FT'] == 1].groupby('DPTO').size()

In [0]:
# Tabla con la tasa de desempleo
tabla_mujeres = pd.DataFrame({
    'PEA': pea_mujeres,
    'DESEMPLEADAS': desemp_mujeres,
    'CASOS_MUESTRA': casos_pea
})
tabla_mujeres['DESEMPLEADAS'] = tabla_mujeres['DESEMPLEADAS'].fillna(0)
tabla_mujeres['TD'] = tabla_mujeres['DESEMPLEADAS'] / tabla_mujeres['PEA'] * 100

tabla_mujeres = tabla_mujeres.reset_index()
tabla_mujeres['Departamento'] = tabla_mujeres['DPTO'].map(dpto_nombre)
tabla_mujeres = tabla_mujeres[['Departamento', 'PEA', 'DESEMPLEADAS', 'CASOS_MUESTRA', 'TD']]

In [0]:
tabla_mujeres.sort_values('TD')

In [0]:
# Los 3 departamentos con menor tasa de desempleo
tabla_mujeres.sort_values('TD', ascending=True).head(3)

5. Calcule el promedio de ingresos laborales (variable INGLABO) de las personas 
residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle 
del Cauca.

In [0]:
# Cargar ocupados de junio y anexar sexo/edad
ocupados_junio = (spark.read.format("delta")
                  .table("workspace.electiva.ocupados")
                  .filter("PERIODO = 20260626")
                  .toPandas()
                  .merge(cgse_junio, on=llaves, how='left'))

# Verificar que la variable esté en la tabla
print('INGLABO' in ocupados_junio.columns, 'DPTO' in ocupados_junio.columns)

dptos_interes = [5, 17, 63, 66, 76]

ocup_5 = ocupados_junio[ocupados_junio['DPTO'].isin(dptos_interes)].copy()

In [0]:
# Asegurar que sea numérica (desde Spark a veces llega como texto o Decimal)
ocup_5['INGLABO'] = pd.to_numeric(ocup_5['INGLABO'], errors='coerce')

In [0]:
# Solo personas con ingreso laboral reportado
ocup_5 = ocup_5[ocup_5['INGLABO'].notna() & (ocup_5['INGLABO'] > 0)]

In [0]:
# Ingreso ponderado: INGLABO x factor de expansión
ocup_5['ING_POND'] = ocup_5['INGLABO'] * ocup_5['FEX_C18']

g = ocup_5.groupby('DPTO')
tabla_ingresos = pd.DataFrame({
    'ING_TOTAL': g['ING_POND'].sum(),
    'OCUPADOS': g['FEX_C18'].sum(),
    'CASOS_MUESTRA': g.size()
})
tabla_ingresos['PROMEDIO_INGLABO'] = tabla_ingresos['ING_TOTAL'] / tabla_ingresos['OCUPADOS']

tabla_ingresos = tabla_ingresos.reset_index()
tabla_ingresos['Departamento'] = tabla_ingresos['DPTO'].map(dpto_nombre)
tabla_ingresos = tabla_ingresos[['Departamento', 'OCUPADOS', 'CASOS_MUESTRA', 'PROMEDIO_INGLABO']]
tabla_ingresos = tabla_ingresos.sort_values('PROMEDIO_INGLABO', ascending=False)

In [0]:
# Promedio conjunto de los 5 departamentos (opcional)
prom_total = ocup_5['ING_POND'].sum() / ocup_5['FEX_C18'].sum()

In [0]:
print(tabla_ingresos.to_string(index=False, formatters={
    'OCUPADOS': '{:,.0f}'.format, 'PROMEDIO_INGLABO': '${:,.0f}'.format}))
print(f"\nPromedio conjunto de los 5 departamentos: ${prom_total:,.0f}")